# Bounded Queue Data Structure, $O(N)$ vs $O(1)$ Mechanics, and Circular Ring Buffers

**Session:** 2026-11-03 · OOP Practice Lab  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

A **Queue** is a linear abstract data type operating on the **First-In, First-Out (FIFO)** principle: elements are inserted at the rear and removed from the front.

In this notebook, we implement and evaluate:
1. Bounded `Queue` using a naive Python list (`pop(0)`), analyzing its $O(N)$ memory shifting cost.
2. High-performance `CircularQueue` implementing a circular ring buffer with modulo index arithmetic ($\text{rear} = (\text{rear} + 1) \pmod C$), achieving deterministic $O(1)$ enqueue and dequeue operations.
3. Micro-benchmark profiling throughput across queue sizes.
4. Telemetry workload processing from `../data/buffer_workload.json`.


## 1. Naive List-Based Queue Implementation

Conforming directly to the lecture slide specification using a standard list container.


In [ ]:
from typing import Any


class NaiveQueue:
    """Bounded FIFO Queue implemented using a Python list (demonstrating O(N) dequeue)."""

    def __init__(self, capacity: int) -> None:
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError(f"Capacity must be a positive integer, got {capacity}")
        self.__capacity: int = int(capacity)
        self.__queue: list[Any] = []

    def is_empty(self) -> bool:
        """Return True if queue contains zero elements."""
        return len(self.__queue) == 0

    def is_full(self) -> bool:
        """Return True if queue has reached maximum capacity."""
        return len(self.__queue) == self.__capacity

    def enqueue(self, value: Any) -> None:
        """Add element to the rear of the queue.

        Raises:
            OverflowError: If queue is at capacity.
        """
        if self.is_full():
            raise OverflowError(f"Queue Overflow: capacity {self.__capacity} reached.")
        self.__queue.append(value)

    def dequeue(self) -> Any:
        """Remove and return the front element of the queue.

        Raises:
            IndexError: If queue is empty.
        """
        if self.is_empty():
            raise IndexError("Queue Underflow: cannot dequeue from an empty queue.")
        return self.__queue.pop(0)  # O(N) memory shift!

    def front(self) -> Any:
        """Return the front element without removing it."""
        if self.is_empty():
            raise IndexError("Queue Underflow: cannot inspect front of an empty queue.")
        return self.__queue[0]

    def __len__(self) -> int:
        return len(self.__queue)


# Verify basic FIFO behavior
nq = NaiveQueue(capacity=3)
nq.enqueue("A")
nq.enqueue("B")
nq.enqueue("C")
assert nq.is_full() is True
assert nq.dequeue() == "A"
assert nq.dequeue() == "B"
assert nq.front() == "C"
assert nq.dequeue() == "C"
assert nq.is_empty() is True
print("NaiveQueue FIFO verified.")

## 2. High-Performance Circular Ring Buffer Queue ($O(1)$)

To eliminate the $O(N)$ linear memory copy incurred by `list.pop(0)`, we allocate a fixed array and update pointer offsets using modulo arithmetic:
$$\text{rear} \leftarrow (\text{rear} + 1) \pmod C$$
$$\text{front} \leftarrow (\text{front} + 1) \pmod C$$


In [ ]:
class CircularQueue:
    """Optimal O(1) Bounded Queue utilizing a fixed circular array ring buffer."""

    def __init__(self, capacity: int) -> None:
        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError(f"Capacity must be a positive integer, got {capacity}")
        self.__capacity: int = int(capacity)
        self.__buffer: list[Any] = [None] * self.__capacity
        self.__front: int = 0
        self.__rear: int = 0
        self.__count: int = 0

    def is_empty(self) -> bool:
        """Check if the circular queue has zero elements."""
        return self.__count == 0

    def is_full(self) -> bool:
        """Check if the circular queue has reached capacity."""
        return self.__count == self.__capacity

    def enqueue(self, value: Any) -> None:
        """Insert value at rear pointer in O(1) deterministic time."""
        if self.is_full():
            raise OverflowError(f"Circular Queue Overflow: capacity {self.__capacity} reached.")
        self.__buffer[self.__rear] = value
        self.__rear = (self.__rear + 1) % self.__capacity
        self.__count += 1

    def dequeue(self) -> Any:
        """Remove and return value from front pointer in O(1) deterministic time."""
        if self.is_empty():
            raise IndexError("Circular Queue Underflow: cannot dequeue from an empty queue.")
        value = self.__buffer[self.__front]
        self.__buffer[self.__front] = None  # Release reference for garbage collection
        self.__front = (self.__front + 1) % self.__capacity
        self.__count -= 1
        return value

    def front(self) -> Any:
        """Inspect front element without removal in O(1) time."""
        if self.is_empty():
            raise IndexError("Circular Queue Underflow: cannot inspect front of an empty queue.")
        return self.__buffer[self.__front]

    def __len__(self) -> int:
        return self.__count

    def __repr__(self) -> str:
        return f"CircularQueue(capacity={self.__capacity}, count={self.__count})"


# Verify circular wrap-around behavior
cq = CircularQueue(capacity=3)
cq.enqueue(10)
cq.enqueue(20)
cq.enqueue(30)
assert cq.is_full() is True

# Dequeue 2 items (front moves from 0 -> 1 -> 2)
assert cq.dequeue() == 10
assert cq.dequeue() == 20

# Enqueue 2 new items (rear wraps around modulo 3)
cq.enqueue(40)
cq.enqueue(50)
assert cq.is_full() is True

# Verify correct retrieval order across wrap boundary
assert cq.dequeue() == 30
assert cq.dequeue() == 40
assert cq.dequeue() == 50
assert cq.is_empty() is True
print("CircularQueue modulo wrap-around verified successfully.")

## 3. Micro-Benchmarking: Naive List vs. Circular Ring Buffer

We benchmark $10{,}000$ enqueue-dequeue cycles on a moderately sized queue ($N = 5{,}000$) to observe the asymptotic performance difference.


In [ ]:
import time

CAPACITY = 5000
OPERATIONS = 10000

# Benchmark Naive Queue
naive = NaiveQueue(capacity=CAPACITY)
# Pre-fill
for i in range(CAPACITY - 1):
    naive.enqueue(i)

t0 = time.perf_counter()
for i in range(OPERATIONS):
    naive.enqueue(i)
    naive.dequeue()
t_naive = time.perf_counter() - t0

# Benchmark Circular Queue
circ = CircularQueue(capacity=CAPACITY)
for i in range(CAPACITY - 1):
    circ.enqueue(i)

t0 = time.perf_counter()
for i in range(OPERATIONS):
    circ.enqueue(i)
    circ.dequeue()
t_circ = time.perf_counter() - t0

speedup = t_naive / max(t_circ, 1e-9)
print(f"Naive Queue (list.pop(0)): {t_naive:.4f} seconds")
print(f"Circular Ring Queue:       {t_circ:.4f} seconds")
print(f"Circular Queue is {speedup:.1f}x faster!")
assert t_circ < t_naive, "Circular queue should be significantly faster than list.pop(0)"

## 4. Workload Pipeline Simulation

We ingest tasks from `../data/buffer_workload.json` and simulate processing via `CircularQueue`.


In [ ]:
import json
from pathlib import Path

workload_path = Path("../data/buffer_workload.json")
assert workload_path.is_file(), f"Workload file not found: {workload_path}"

with workload_path.open("r", encoding="utf-8") as f:
    workload = json.load(f)

task_buffer = CircularQueue(capacity=len(workload) + 2)

for item in workload:
    task_buffer.enqueue(item)

print(f"Enqueued {len(task_buffer)} computational tasks.")
assert task_buffer.front()["task_id"] == "task-001"

processed_tasks = []
total_latency = 0.0
while not task_buffer.is_empty():
    task = task_buffer.dequeue()
    processed_tasks.append(task["task_id"])
    total_latency += task["latency_ms"]

print(f"Processed order: {processed_tasks}")
print(f"Cumulative Pipeline Latency: {total_latency:.1f} ms")
assert len(processed_tasks) == len(workload)